# Getting a Kernel ready...

In [ ]:
#r "nuget: Microsoft.DotNet.Interactive.SqlServer, *-*"

In [ ]:
#!connect mssql --kernel-name sql2025 --connection-string "Server=sql-ctp21;TrustServerCertificate=True;Integrated Security=True"

# Check our connection

In [ ]:
SELECT @@VERSION

# Create a sample DB and get it ready

In [ ]:
USE master
IF EXISTS (SELECT name FROM sys.databases WHERE name='DemoDB')BEGIN
    ALTER DATABASE DemoDB SET SINGLE_USER WITH ROLLBACK IMMEDIATE;
    DROP DATABASE DemoDB;
END
GO
CREATE DATABASE DemoDB
GO
USE DemoDB
GO

In [ ]:
create master key encryption by password = 'MyTest!Mast3rP4ss'


In [ ]:
sp_configure 'external rest endpoint enabled', 1;
RECONFIGURE WITH OVERRIDE

# Try some first REST APIs

In [ ]:
DECLARE @response NVARCHAR(max)

EXEC Sp_invoke_external_rest_endpoint
  @url = N'https://datausa.io/api/data?drilldowns=State&measures=Population&year=latest',
  @method = 'GET',
  @response = @response output;

SELECT TOP 5 * FROM Openjson(Json_query(@response, '$.result.data')) A 

In [ ]:
DECLARE @response NVARCHAR(max)

EXEC Sp_invoke_external_rest_endpoint
  @url = N'https://reqres.in/api/products/',
  @headers = '{"x-api-key":"reqres-free-v1"}',
  @method = 'GET',
  @response = @response output;

SELECT * FROM   Openjson(Json_query(@response, '$.result.data')) A 

In [ ]:
CREATE DATABASE SCOPED CREDENTIAL [https://reqres.in/api/]
WITH IDENTITY = 'HTTPEndpointHeaders', SECRET = '{"x-api-key":"reqres-free-v1"}';


In [ ]:
DECLARE @response NVARCHAR(max)

EXEC Sp_invoke_external_rest_endpoint
  @url = N'https://reqres.in/api/products/',
  @credential = [https://reqres.in/api/],
  @method = 'GET',
  @response = @response output;

SELECT * FROM   Openjson(Json_query(@response, '$.result.data')) A 

In [ ]:
DECLARE @response NVARCHAR(max)

EXEC Sp_invoke_external_rest_endpoint
  @url = N'https://reqres.in/api/products/',
  @credential = [https://reqres.in/api/],
  @method = 'GET',
  @response = @response output;

SELECT Json_value(value, '$.name')          name,
       Json_value(value, '$.color')         color,
       Json_value(value, '$.pantone_value') pantone_value
FROM   Openjson(Json_query(@response, '$.result.data'), 'strict $') AS export 

In [ ]:
DECLARE @apiKey NVARCHAR(255) = 'XXX'
DECLARE @url NVARCHAR(max) =  N'https://api.exchangeratesapi.io/v1/latest?access_key=' + @apikey + '&base=EUR&symbols=USD,GBP'
DECLARE @response NVARCHAR(max)

EXEC Sp_invoke_external_rest_endpoint
  @url = @url,
  @response = @response output

SELECT [key],
       value
FROM   Openjson(Json_query(@response, '$.result.rates'), 'strict $') AS export 

In [ ]:
CREATE DATABASE SCOPED CREDENTIAL [https://api.exchangeratesapi.io/]
WITH IDENTITY = 'HTTPEndpointHeaders', SECRET = '{"access_key":"XXX"}';

In [ ]:
DECLARE @url NVARCHAR(max) = N'https://api.exchangeratesapi.io/v1/latest?base=EUR&symbols=USD,GBP'
DECLARE @response NVARCHAR(max)

EXEC Sp_invoke_external_rest_endpoint
  @credential = [https://api.exchangeratesapi.io/],
  @url = @url,
  @response = @response output

SELECT [key],
       value
FROM   Openjson(Json_query(@response, '$.result'), 'strict $') AS export 

In [ ]:
ALTER DATABASE SCOPED CREDENTIAL [https://api.exchangeratesapi.io/]
WITH IDENTITY = 'HttpEndpointQueryString', SECRET = '{"access_key":"XXX"}';

In [ ]:
DECLARE @url NVARCHAR(max) = N'https://api.exchangeratesapi.io/v1/latest?base=EUR&symbols=USD,GBP'
DECLARE @response NVARCHAR(max)

EXEC Sp_invoke_external_rest_endpoint
  @credential = [https://api.exchangeratesapi.io/],
  @url = @url,
  @response = @response output

SELECT [key],
       value
FROM   Openjson(Json_query(@response, '$.result.rates'), 'strict $') AS export 

# API Response Codes

In [ ]:
DECLARE @Result int
DECLARE @url NVARCHAR(max) = N'https://api.exchangeratesapi.io/v1/latest?base=EUR&symbols=USD,GBP'
DECLARE @response NVARCHAR(max)

EXEC @Result =  Sp_invoke_external_rest_endpoint
  @credential = [https://api.exchangeratesapi.io/],
  @url = @url,
  @response = @response output

SELECT @result,@response

In [ ]:
DECLARE @Result int
DECLARE @url NVARCHAR(max) = N'https://api.exchangeratesapi.io/v1/latest?base=EUR&symbols=USD,GBP'
DECLARE @response NVARCHAR(max)

EXEC @Result =  Sp_invoke_external_rest_endpoint
  @url = @url,
  @response = @response output

SELECT @result,@response

# OAuth 
(We'll talk about basic auth, managed identity etc. another time)

In [ ]:
IF EXISTS (SELECT * FROM sys.database_scoped_credentials WHERE NAME = 'https://test.api.amadeus.com/')
  BEGIN
      DROP DATABASE scoped credential [https://test.api.amadeus.com/];
  END

GO

DECLARE @response NVARCHAR(max)

EXEC Sp_invoke_external_rest_endpoint
  @url = N'https://test.api.amadeus.com/v1/security/oauth2/token',
  @headers='{"content-type":"application/x-www-form-urlencoded"}',
  @payload=N'grant_type=client_credentials&client_id=XXX&dummy=XXX&client_secret=XXX',
  @method = 'POST',
  @response = @response output;

DECLARE @secret NVARCHAR(max)

SELECT @secret = value
FROM   Openjson(Json_query(@response, '$.result')) A
WHERE  [key] = 'access_token'

SET @secret = N'{"Authorization":"Bearer ' + @secret + '"}'
SET @secret = N'CREATE DATABASE SCOPED CREDENTIAL [https://test.api.amadeus.com/] WITH IDENTITY = ''HTTPEndpointHeaders'', SECRET = ''' + @secret + ''''

SELECT @secret

EXEC Sp_executesql  @secret 

In [ ]:
DECLARE @response NVARCHAR(max)
DECLARE @From NVARCHAR(3) = 'CGN'
DECLARE @To NVARCHAR(3) = 'NUE'
DECLARE @Date NVARCHAR(10) = CONVERT(NVARCHAR, Dateadd(d, 1, Getdate()), 23)
DECLARE @url NVARCHAR(max) = N'https://test.api.amadeus.com/v2/shopping/flight-offers?originLocationCode=' + @From + '&destinationLocationCode=' + @To + '&departureDate=' + @Date + '&adults=1&travelClass=BUSINESS&includedAirlineCodes=LH,UA'

EXEC Sp_invoke_external_rest_endpoint
  @url = @url,
  @credential = [https://test.api.amadeus.com/],
  @method = 'GET',
  @response = @response output;

SELECT Json_value(value, '$.price.grandTotal')                                                     Price,
       Replace(Json_value(value, '$.itineraries[0].duration'), 'PT', '')                           Duration,
       Replace(Json_query(value, '$.itineraries[0].segments[*].arrival.iataCode'), ','+ @To, '')   Stops
FROM   Openjson(Json_query(@response, '$.result.data'))
ORDER  BY duration 

In [ ]:
IF EXISTS (SELECT * FROM sys.database_scoped_credentials WHERE NAME = 'https://management.azure.com/')
  BEGIN
      DROP DATABASE scoped credential [https://management.azure.com/];
  END

GO

DECLARE @response NVARCHAR(max)

EXEC Sp_invoke_external_rest_endpoint
  @url = N'https://login.microsoftonline.com/XXXX/oauth2/token',
  @headers='{"content-type":"application/x-www-form-urlencoded"}',
  @payload=N'grant_type=client_credentials&client_id=XXX&client_secret=XXX&resource=https%3A%2F%2Fmanagement.azure.com%2F',
  @method = 'POST',
  @response = @response output;

DECLARE @secret NVARCHAR(max)

SELECT @secret = value
FROM   Openjson(Json_query(@response, '$.result')) A
WHERE  [key] = 'access_token'

SET @secret = N'{"Authorization":"Bearer ' + @secret + '"}'
SET @secret = N'CREATE DATABASE SCOPED CREDENTIAL [https://management.azure.com/] WITH IDENTITY = ''HTTPEndpointHeaders'', SECRET = ''' + @secret + ''''

EXEC Sp_executesql  @secret 

In [ ]:
DECLARE @response NVARCHAR(max)

EXEC Sp_invoke_external_rest_endpoint @url = 'https://management.azure.com/subscriptions/XXX/providers/Microsoft.Compute/disks?api-version=2025-01-02',
@credential = [https://management.azure.com/],
@method = 'GET',
@response = @response output;

SELECT *
FROM   (SELECT Json_value(value, '$.name')            DiskName,
               Json_value(value, '$.properties.tier') DiskTier
        FROM   Openjson(Json_query(@response, '$.result.value'))) A
WHERE  diskname LIKE 'SQL2025%' 

## Chatting with AI

In [ ]:
DECLARE @response NVARCHAR(max)
DECLARE @prompt NVARCHAR(max) = N'what is the airspeed velocity of an unladen swallow?'
DECLARE @model NVARCHAR(250) = N'smollm2'
DECLARE @payload NVARCHAR(max) = N'{"model":"' + @model + '","prompt":"' + @prompt + '","stream": false }'

EXEC Sp_invoke_external_rest_endpoint
  @url = N'https://ai-gpu.lab.bwdemo.io:443/api/generate',
  @payload = @payload,
  @timeout = 230,
  @response = @response output;

SELECT value
FROM   Openjson(Json_query(@response, '$.result')) A
WHERE  [key] = 'response' 

In [ ]:
IF OBJECT_ID('dbo.ConferenceSessions', 'U') IS NOT NULL DROP TABLE dbo.ConferenceSessions;
CREATE TABLE [dbo].[ConferenceSessions] ([id] int NOT NULL,
[Title] [nvarchar](4000) NULL,
[description] [nvarchar](4000) NULL,
[startsat] datetime NULL,
[endsat] datetime NULL,
MainSpeaker [nvarchar](4000) NULL,
[Speakers] [nvarchar](4000) NULL) ON [PRIMARY]

In [ ]:
declare @URL nvarchar(500) ='https://sessionize.com/api/v2/0scvywi2/view/sessions'
declare @response nvarchar(max)

exec sp_invoke_external_rest_endpoint @url=@URL, @response=@response OUTPUT
truncate table ConferenceSessions
insert into ConferenceSessions
select id, max(title) Title, max(description) description, max(startsat) startsat, max(endsat) endsat, max(Name) MainSpeaker, STRING_AGG(Name, ', ') Speaker
from(SELECT JSON_VALUE(value, '$.id') id, JSON_VALUE(value, '$.title') title, JSON_VALUE(value, '$.description') description, JSON_VALUE(value, '$.startsAt') startsat, JSON_VALUE(value, '$.endsAt') endsat, speakers.Name
     FROM OPENJSON(JSON_QUERY(@response, '$.result[0].sessions'), 'strict $') as export
          OUTER APPLY
         OPENJSON(JSON_QUERY(value, '$.speakers'))
         WITH(name NVARCHAR(50) '$.name')speakers)a
group by id

In [ ]:
SELECT top 3 * FROM ConferenceSessions

In [ ]:
DECLARE @prompt NVARCHAR(max)

SELECT TOP 1 @prompt = 'You are at a conference called Data Saturday Rheinland in St Augustin and attended the session below. Please write a Linkedin post about what you liked about the session. '
          + mainspeaker + ' -' + title + ' -' + [description]
FROM   ConferenceSessions
WHERE  speakers LIKE '%weissman%' order by id desc

SELECT @prompt 

In [ ]:
DECLARE @prompt NVARCHAR(max)

SELECT TOP 1 @prompt = 'You are at a conference called Data Saturday Rheinland in St Augustin and attended the session below. Please write a Linkedin post about what you liked about the session. '
          + mainspeaker + ' -' + title + ' -' + [description]
FROM   ConferenceSessions
WHERE  speakers LIKE '%weissman%' order by id desc

SET @prompt = Replace(Replace(@prompt, Char(13), ''), Char(10), '')

DECLARE @response NVARCHAR(max)
DECLARE @model NVARCHAR(250) = N'smollm2'
DECLARE @payload NVARCHAR(max) = N'{"model":"' + @model + '","prompt":"' + @prompt + '","stream": false }'

EXEC Sp_invoke_external_rest_endpoint
  @url = N'https://ai-gpu.lab.bwdemo.io:443/api/generate',
  @payload = @payload,
  @timeout = 230,
  @response = @response output;

SELECT value FROM   Openjson(Json_query(@response, '$.result')) A WHERE  [key] = 'response' 

# Vector Distances

In [ ]:
CREATE TABLE [dbo].[speakers] ([fullName] [nvarchar](4000) NULL,
[bio] [nvarchar](4000) NULL) ON [PRIMARY]

In [ ]:
declare @URL nvarchar(500) ='https://sessionize.com/api/v2/0scvywi2/view/speakers'
declare @response nvarchar(max)
exec sp_invoke_external_rest_endpoint @url=@URL, @response=@response OUTPUT
truncate table speakers
insert into speakers
SELECT JSON_VALUE(value, '$.fullName') fullName, JSON_VALUE(value, '$.bio') bio
FROM OPENJSON(JSON_QUERY(@response, '$.result'), 'strict $') as export

In [ ]:
SELECT TOP 3 Speakers,Title,description FROM ConferenceSessions where description like '%chatbot%'


In [ ]:
SELECT TOP 3 Speakers,Title,description FROM ConferenceSessions where description like '%AI%'

In [ ]:
ALTER TABLE ConferenceSessions
ADD embeddings VECTOR(768),embeddings_full VECTOR(768),embeddings_speaker VECTOR(768)

In [ ]:
CREATE EXTERNAL MODEL ollama
WITH (
    LOCATION = 'https://ai-gpu.lab.bwdemo.io:443/api/embed',
    API_FORMAT = 'ollama',
    MODEL_TYPE = EMBEDDINGS,
    MODEL = 'nomic-embed-text'
);

In [ ]:
CREATE EXTERNAL MODEL ollamacpu
WITH (
    LOCATION = 'https://ai-cpu.lab.bwdemo.io:443/api/embed',
    API_FORMAT = 'ollama',
    MODEL_TYPE = EMBEDDINGS,
    MODEL = 'nomic-embed-text'
);

In [ ]:
SELECT AI_GENERATE_EMBEDDINGS(N'BDC forever!', ollama)   

In [ ]:
UPDATE ConferenceSessions SET DESCRIPTION = '' WHERE DESCRIPTION IS NULL

In [ ]:
update ConferenceSessions set embeddings = AI_GENERATE_EMBEDDINGS(title + ' - ' + description,ollama)

In [ ]:
update ConferenceSessions set embeddings = AI_GENERATE_EMBEDDINGS(title + ' - ' + description,ollamacpu)

In [ ]:
SELECT title,description INTO MoreSessions FROM ConferenceSessions
GO
INSERT INTO MoreSessions SELECT title,description FROM MoreSessions
GO
INSERT INTO MoreSessions SELECT title,description FROM MoreSessions
GO
SELECT COUNT(*) FROM MoreSessions

In [ ]:
ALTER TABLE MoreSessions
ADD embeddings VECTOR(768)

In [ ]:
update MoreSessions set embeddings = AI_GENERATE_EMBEDDINGS(title + ' - ' + description,ollamacpu)

In [ ]:
update MoreSessions set embeddings = AI_GENERATE_EMBEDDINGS(title + ' - ' + description,ollama)

In [ ]:
DROP TABLE MoreSessions

In [ ]:
UPDATE A SET [embeddings_speaker] = AI_GENERATE_EMBEDDINGS( ISNULL(bio, ''), ollama)       FROM ConferenceSessions a         LEFT OUTER JOIN speakers b        ON a.mainspeaker = b.fullname where embeddings_speaker is null
UPDATE A SET [embeddings_full] = AI_GENERATE_EMBEDDINGS(speakers + ' (' + ISNULL(bio, '') + '): ' + title + ' - ' + description, ollama )       FROM ConferenceSessions a         LEFT OUTER JOIN speakers b        ON a.mainspeaker = b.fullname where embeddings_full is null

In [ ]:
SELECT Speakers,Title,description FROM ConferenceSessions where description like '%chatbots%'

In [ ]:
DECLARE @search_text NVARCHAR(MAX) = 'chatbots';
DECLARE @search_vector VECTOR(768) = AI_GENERATE_EMBEDDINGS(@search_text,ollama);
 
SELECT TOP 1 Speakers,title,description,
    vector_distance('cosine', @search_vector, p.embeddings) AS distance
FROM ConferenceSessions p WHERE Description <> ''
ORDER BY distance;

In [ ]:
DECLARE @search_text NVARCHAR(MAX) = 'chatbots';
DECLARE @search_vector VECTOR(768) = AI_GENERATE_EMBEDDINGS(@search_text,ollama);
 
SELECT TOP 1 Speakers,title,vector_distance('cosine', @search_vector, p.embeddings) AS distance
FROM ConferenceSessions p WHERE Description <> ''
ORDER BY distance;

SELECT TOP 1 Speakers,title,vector_distance('dot', @search_vector, p.embeddings) AS distance
FROM ConferenceSessions p WHERE Description <> ''
ORDER BY distance;

SELECT TOP 1 Speakers,title,vector_distance('euclidean', @search_vector, p.embeddings) AS distance
FROM ConferenceSessions p WHERE Description <> ''
ORDER BY distance;

In [ ]:
DECLARE @search_text NVARCHAR(MAX) = 'I am looking for a session by a speaker from the netherlands or belgium';
DECLARE @search_vector VECTOR(768) = AI_GENERATE_EMBEDDINGS(@search_text,ollama);

SELECT TOP(3)
    Speakers,
    vector_distance('cosine', @search_vector, p.embeddings) AS distance
FROM ConferenceSessions p
ORDER BY distance;

SELECT TOP(3)
    Speakers,
    vector_distance('cosine', @search_vector, p.embeddings_full) AS distance
FROM ConferenceSessions p
ORDER BY distance;

SELECT TOP(3)
    Speakers,
    vector_distance('cosine', @search_vector, p.embeddings_speaker) AS distance
FROM ConferenceSessions p
ORDER BY distance;

In [ ]:
CREATE EXTERNAL MODEL ollama2
WITH (
    LOCATION = 'https://ai-gpu.lab.bwdemo.io:443/api/embed',
    API_FORMAT = 'Ollama',
    MODEL_TYPE = EMBEDDINGS,
    MODEL = 'mxbai-embed-large'
);

In [ ]:
DECLARE @search_text NVARCHAR(MAX) = 'I am looking for a session by a speaker from the netherlands or belgium';
DECLARE @search_vector VECTOR(768) = AI_GENERATE_EMBEDDINGS(@search_text,ollama2);

In [ ]:
ALTER TABLE ConferenceSessions
ADD embeddings_2 VECTOR(1024),embeddings_speaker_2 VECTOR(1024)

In [ ]:
update ConferenceSessions set embeddings_2 = AI_GENERATE_EMBEDDINGS(title + ' - ' + description,ollama2) where embeddings_2 is null
UPDATE A SET [embeddings_speaker_2] = AI_GENERATE_EMBEDDINGS( ISNULL(bio, ''),ollama2)       FROM ConferenceSessions a         LEFT OUTER JOIN speakers b        ON a.mainspeaker = b.fullname where embeddings_speaker_2 is null

In [ ]:
DECLARE @search_text NVARCHAR(MAX) = 'I am looking for a session by a speaker from the netherlands or belgium';
DECLARE @search_vector VECTOR(768) = AI_GENERATE_EMBEDDINGS(@search_text,ollama);
DECLARE @search_vector_2 VECTOR(1024) = AI_GENERATE_EMBEDDINGS(@search_text,ollama2);

SELECT TOP(3)
    Speakers,
    vector_distance('cosine', @search_vector, p.embeddings_speaker) AS distance
FROM ConferenceSessions p
ORDER BY distance;

SELECT TOP(3)
    Speakers,
    vector_distance('cosine', @search_vector_2, p.embeddings_speaker_2) AS distance
FROM ConferenceSessions p
ORDER BY distance;

In [ ]:
DECLARE @search_text NVARCHAR(MAX) = 'chatbots';
DECLARE @search_vector VECTOR(768) = AI_GENERATE_EMBEDDINGS(@search_text,ollama);
DECLARE @search_vector_2 VECTOR(1024) = AI_GENERATE_EMBEDDINGS(@search_text,ollama2);
 
SELECT TOP(2)
    Speakers,title,description,
    vector_distance('cosine', @search_vector, p.embeddings) AS distance
FROM ConferenceSessions p WHERE Description <> ''
ORDER BY distance;

SELECT TOP(2)
    Speakers,title,description,
    vector_distance('cosine', @search_vector_2, p.embeddings_2) AS distance
FROM ConferenceSessions p WHERE Description <> ''
ORDER BY distance;

In [ ]:
DECLARE @search_text NVARCHAR(MAX) = 'chatbots';
DECLARE @search_vector VECTOR(768) = AI_GENERATE_EMBEDDINGS(@search_text,ollama);

 
SELECT TOP(2)
    Speakers,title,description,
    vector_distance('cosine', @search_vector, p.embeddings) AS distance
FROM ConferenceSessions p WHERE Description <> ''
ORDER BY distance;


SET  @search_text = 'i want to learn about chatbots';
SET @search_vector = AI_GENERATE_EMBEDDINGS(@search_text,ollama);
SELECT TOP(2)
    Speakers,title,description,
    vector_distance('cosine', @search_vector, p.embeddings) AS distance
FROM ConferenceSessions p WHERE Description <> ''
ORDER BY distance;

# Vector Indexes

In [ ]:
DBCC TRACEON (466, 474, 13981, -1);
DBCC TRACESTATUS;

In [ ]:
ALTER TABLE ConferenceSessions
ADD CONSTRAINT PK_a_Id PRIMARY KEY (Id)

In [ ]:
CREATE VECTOR INDEX vec_idx ON ConferenceSessions([embeddings])
WITH (
    metric = 'cosine',
    type = 'diskann',
    maxdop = 8
);

In [ ]:
UPDATE ConferenceSessions set title = title + '!'

In [ ]:
SELECT Title,description FROM ConferenceSessions where title like '%PBI%' or description like '%PBI%'

In [ ]:
DECLARE @search_text NVARCHAR(MAX) = 'PBI';
DECLARE @search_vector VECTOR(768) = AI_GENERATE_EMBEDDINGS(@search_text,ollama);

SELECT
    Speakers,title,description,
    s.distance
FROM vector_search(
    table = ConferenceSessions AS t,
    column = [embeddings],
    similar_to = @search_vector,
    metric = 'cosine',
    top_n = 5
) AS s
ORDER BY s.distance;

In [ ]:
DECLARE @search_text NVARCHAR(MAX) = 'PBI';
DECLARE @search_vector VECTOR(768) = AI_GENERATE_EMBEDDINGS(@search_text,ollama);

SELECT
    Speakers,title,description,
    s.distance
FROM vector_search(
    table = ConferenceSessions AS t,
    column = [embeddings],
    similar_to = @search_vector,
    metric = 'cosine',
    top_n = 3
) AS s
ORDER BY s.distance;

SELECT TOP(3)
    Speakers,title,description,
    vector_distance('cosine', @search_vector, p.embeddings) AS distance
FROM ConferenceSessions p WHERE Description <> ''
ORDER BY distance;

In [ ]:
DECLARE @search_text NVARCHAR(MAX) = 'Ich suche nach einer legendären session';
DECLARE @search_vector VECTOR(768) = AI_GENERATE_EMBEDDINGS(@search_text,ollama);

SELECT
    Speakers,title,
    s.distance
FROM vector_search(
    table = ConferenceSessions AS t,
    column = [embeddings],
    similar_to = @search_vector,
    metric = 'cosine',
    top_n = 1
) AS s
ORDER BY s.distance;

# And now something bigger..

In [ ]:
#!connect mssql --kernel-name sql2025-stackoverflow --connection-string "Server=sqlstackoverflow\ctp21;TrustServerCertificate=True;Integrated Security=True"

In [ ]:
use StackOverflowExcerpt

In [ ]:
SET STATISTICS IO ON;
SET STATISTICS TIME ON;

In [ ]:
DECLARE @search_text NVARCHAR(MAX) = 'i want to build a website using visual basic and active server pages';
DECLARE @search_vector VECTOR(768) = AI_GENERATE_EMBEDDINGS(@search_text,ollama);
 
SELECT top 2
    id,title,body,
    vector_distance('cosine', @search_vector, p.embeddings) AS distance
FROM Posts_Excerpt_small p
 
ORDER BY distance;

In [ ]:
DECLARE @search_text NVARCHAR(MAX) = 'i want to build a website using visual basic and active server pages';
DECLARE @search_vector VECTOR(768) = AI_GENERATE_EMBEDDINGS(@search_text,ollama);

SELECT
    t.id,title,body,    s.distance
FROM vector_search(
    table = Posts_Excerpt_small AS t,
    column = [embeddings],
    similar_to = @search_vector,
    metric = 'cosine',
    top_n = 2
) AS s  
ORDER BY s.distance;

In [ ]:
DECLARE @search_text NVARCHAR(MAX) = 'i want to build a website using visual basic and active server pages';
DECLARE @search_vector VECTOR(768) = AI_GENERATE_EMBEDDINGS(@search_text,ollama);
 
SELECT top 2
    id,title,body,
    vector_distance('cosine', @search_vector, p.embeddings) AS distance
FROM Posts_Excerpt_large p
 
ORDER BY distance;

In [ ]:
DECLARE @search_text NVARCHAR(MAX) = 'i want to build a website using visual basic and active server pages';
DECLARE @search_vector VECTOR(768) = AI_GENERATE_EMBEDDINGS(@search_text,ollama);

SELECT
    t.id,title,body,    s.distance
FROM vector_search(
    table = Posts_Excerpt_large AS t,
    column = [embeddings],
    similar_to = @search_vector,
    metric = 'cosine',
    top_n = 2
) AS s  
ORDER BY s.distance;